# Домашняя работа 4.1. Вернуть маятник в заданное положение
## Зелёный уровень

Нужно подобрать начальную скорость $s=\dot\theta(0)$ для задачи

$$
\ddot\theta+\theta=0,\qquad \theta(0)=0,\qquad \theta(2)=0.2.
$$

Время безразмерное, угол измеряется в радианах.
Для каждого пробного $s$ решается задача Коши

$$
\dot\theta=\omega,\qquad \dot\omega=-\theta,\qquad
(\theta(0),\omega(0))=(0,s).
$$

Для интегрирования используется RK4. Начальная скорость определяется из условия
$R_h(s)=\theta_h(2;s)-0.2=0$, где индекс $h$ обозначает шаг интегрирования.

[Условие задания](https://github.com/yartmihant/Students6Year2026#readme).
Для запуска нужны NumPy и Matplotlib.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.25})
T = 2.0
target = 0.2
s_exact = target / np.sin(T)


def rhs(state):
    theta, omega = state
    return np.array([omega, -theta])


def integrate(s, h):
    if h <= 0:
        raise ValueError("Шаг должен быть положительным.")
    n = round(T / h)
    if n < 1 or not np.isclose(n * h, T, rtol=0, atol=1e-12):
        raise ValueError("Шаг должен быть положительным и делить T без остатка.")
    t = np.arange(n + 1) * h
    states = np.empty((n + 1, 2))
    states[0] = [0, s]
    for i in range(n):
        state = states[i]
        k1 = rhs(state)
        k2 = rhs(state + h * k1 / 2)
        k3 = rhs(state + h * k2 / 2)
        k4 = rhs(state + h * k3)
        states[i + 1] = state + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return t, states


def residual(s, h):
    return integrate(s, h)[1][-1, 0] - target


def exact(t):
    return target * np.sin(t) / np.sin(T)

## 1. Пристрелочные траектории

Начальный шаг выбран равным $h=0.2$.
Для бисекции используется интервал $[0,0.5]$: невязка на левом конце отрицательна,
на правом — положительна. Точка $(2,0.2)$ отмечает требуемое конечное положение.

In [2]:
h_main = 0.2
left, right = 0.0, 0.5
print(f"R_h(0) = {residual(left, h_main):.8f}")
print(f"R_h(0.5) = {residual(right, h_main):.8f}")
assert residual(left, h_main) * residual(right, h_main) < 0

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
for s in [0.0, 0.1, 0.2, 0.3, 0.5]:
    t, states = integrate(s, h_main)
    ax.plot(t, states[:, 0], ".-", label=f"s = {s:g}", markersize=3)
ax.plot(T, target, "r*", markersize=13, label="Требуемое положение", zorder=5)
ax.set(xlabel="t", ylabel="θ, рад", title="Пробные начальные скорости, h = 0.2")
ax.legend()
plt.show()

## 2. Ньютон и бисекция

В методе Ньютона

$$
s_{n+1}=s_n-\frac{R_h(s_n)}{R_h'(s_n)}.
$$

Производная вычисляется центральной разностью с $\delta=10^{-5}$:

$$
R_h'(s)\approx\frac{R_h(s+\delta)-R_h(s-\delta)}{2\delta}.
$$

Каждая оценка производной требует двух дополнительных вычислений невязки.
Начальное приближение Ньютона — $s_0=0$.
Бисекция каждый раз делит текущий интервал пополам и сохраняет половину со сменой знака.

У обоих методов одинаковый критерий остановки: $|R_h(s)|\leq10^{-12}$.
Счётчик включает начальную невязку Ньютона, пробы для производной,
проверки концов интервала бисекции и проверку последнего приближения.
Вычисления для построения графиков в стоимость подбора не входят.

In [3]:
def newton(R, s0=0.0, tol=1e-12, delta=1e-5):
    s = s0
    calls = 0
    history = []
    for _ in range(30):
        r = R(s)
        calls += 1
        history.append((s, r, calls))
        if abs(r) <= tol:
            return s, np.array(history), calls
        derivative = (R(s + delta) - R(s - delta)) / (2 * delta)
        calls += 2
        if abs(derivative) < 1e-14:
            raise RuntimeError("Производная невязки слишком мала.")
        s -= r / derivative
    raise RuntimeError("Метод Ньютона не сошёлся.")


def bisection(R, a=0.0, b=0.5, tol=1e-12):
    fa, fb = R(a), R(b)
    calls = 2
    if abs(fa) <= tol:
        return a, np.array([(a, fa, 1)]), calls
    if abs(fb) <= tol:
        return b, np.array([(b, fb, 2)]), calls
    if fa * fb >= 0:
        raise ValueError("Невязки на концах должны иметь разные знаки.")
    history = []
    for _ in range(100):
        middle = (a + b) / 2
        fm = R(middle)
        calls += 1
        history.append((middle, fm, calls))
        if abs(fm) <= tol:
            return middle, np.array(history), calls
        if fa * fm < 0:
            b, fb = middle, fm
        else:
            a, fa = middle, fm
    raise RuntimeError("Бисекция не сошлась.")


R = lambda s: residual(s, h_main)
s_newton, history_newton, calls_newton = newton(R)
s_bisect, history_bisect, calls_bisect = bisection(R)
print(f"Точная скорость: {s_exact:.12f}")
print("Метод       s                 |R_h(s)|       Вызовы R")
for name, s, history, calls in [("Ньютон", s_newton, history_newton, calls_newton),
                                ("Бисекция", s_bisect, history_bisect, calls_bisect)]:
    print(f"{name:<12}{s:.12f}    {abs(history[-1, 1]):.3e}    {calls}")

In [4]:
speeds = np.linspace(0, 0.5, 101)
residuals = np.array([residual(s, h_main) for s in speeds])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
axes[0].plot(speeds, residuals, color="gray", label="Численная невязка R_h(s)")
axes[0].axhline(0, color="black", linewidth=0.7)
axes[0].scatter(history_bisect[:, 0], history_bisect[:, 1], s=28,
                color="tab:orange", label="Бисекция", zorder=3)
axes[0].plot(history_newton[:, 0], history_newton[:, 1], "rx--",
             markersize=9, label="Ньютон", zorder=4)
for i, (s, r, _) in enumerate(history_bisect[:4], start=1):
    axes[0].annotate(f"B{i}", (s, r), xytext=(5, -16), textcoords="offset points")
for i, (s, r, _) in enumerate(history_newton):
    axes[0].annotate(f"N{i}", (s, r), xytext=(5, 9), textcoords="offset points", color="red")
axes[0].set(xlabel="Начальная скорость s", ylabel="R_h(s)", title="Приближения методов")
axes[0].legend()
for name, history, color in [("Ньютон", history_newton, "tab:red"),
                             ("Бисекция", history_bisect, "tab:orange")]:
    axes[1].semilogy(history[:, 2], np.abs(history[:, 1]), ".-", color=color, label=name)
axes[1].axhline(1e-12, color="gray", linestyle=":", label="Допуск")
axes[1].set(xlabel="Накопленное число вычислений R", ylabel="|R_h(s)|", title="Сходимость подбора")
axes[1].legend()
plt.show()

## 3. Сравнение с точным решением и уменьшение шага

Точное решение и начальная скорость:

$$
\theta(t)=\frac{0.2\sin t}{\sin2},\qquad s_* = \frac{0.2}{\sin2}.
$$

Расчёт повторяется при $h=0.2,\ 0.1,\ 0.05,\ 0.025$.
Для каждого шага оба метода заново подбирают скорость с теми же начальными
приближениями и допуском. В таблице указаны ошибка скорости и максимальная
ошибка угла в узлах сетки. Траектории и графики ошибок ниже относятся к Ньютону;
для бисекции ошибки также приведены в таблице.

In [5]:
steps = [0.2, 0.1, 0.05, 0.025]
solutions = {}
motion_errors = []
print("h      Метод       |s - s_*|      max |θ_h - θ|   |R_h(s)|      Вызовы R")
for h in steps:
    R = lambda s: residual(s, h)
    for name, method in [("Ньютон", newton), ("Бисекция", bisection)]:
        s, history, calls = method(R)
        t, states = integrate(s, h)
        error = np.max(np.abs(states[:, 0] - exact(t)))
        solutions[h, name] = (s, t, states, history, calls)
        print(f"{h:<7g}{name:<12}{abs(s-s_exact):.3e}      {error:.3e}      "
              f"{abs(history[-1, 1]):.3e}    {calls}")
        if name == "Ньютон":
            motion_errors.append(error)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
fine_t = np.linspace(0, T, 500)
axes[0].plot(fine_t, exact(fine_t), "k-", label="Точное решение")
s, t, states, history, calls = solutions[0.2, "Ньютон"]
axes[0].plot(t, states[:, 0], "o", fillstyle="none", label="RK4, h = 0.2")
axes[0].plot(T, target, "r*", markersize=10)
axes[0].set(xlabel="t", ylabel="θ, рад", title="Численное и точное решения")
for h in steps:
    _, t, states, _, _ = solutions[h, "Ньютон"]
    axes[1].plot(t[1:], np.abs(states[1:, 0] - exact(t[1:])), ".-", label=f"h = {h:g}")
axes[1].set(xlabel="t", ylabel="Абсолютная ошибка угла", title="Ошибка вдоль траектории", yscale="log")
axes[2].loglog(steps, motion_errors, "o-", label="Ошибка в узлах")
axes[2].loglog(steps, motion_errors[0] * (np.array(steps) / steps[0])**4,
               "k--", label="Пропорционально h⁴")
axes[2].set(xlabel="h", ylabel="Максимальная ошибка угла", title="Изменение ошибки с шагом")
axes[2].set_xticks(steps, [str(h) for h in steps])
axes[2].minorticks_off()
for ax in axes:
    ax.legend(fontsize=8)
plt.show()
print("Отношения ошибок при уменьшении шага вдвое:")
print(np.array(motion_errors[:-1]) / np.array(motion_errors[1:]))

## 4. Влияние допуска поиска корня

Здесь шаг интегрирования фиксирован: $h=0.2$.
Бисекция повторяется с допусками от $10^{-2}$ до $10^{-12}$.
Сравниваются невязка численного решения $|R_h|$ и ошибка движения
относительно точного решения. Это разные величины: малая невязка означает,
что конечное условие выполнено для дискретного интегратора.

In [6]:
tolerances = 10.0 ** (-np.arange(2, 13, 2))
tol_errors = []
tol_residuals = []
print("Допуск      |R_h(s)|       max |θ_h - θ|   |s - s_*|      Вызовы R")
for tol in tolerances:
    s, history, calls = bisection(lambda s: residual(s, h_main), tol=tol)
    t, states = integrate(s, h_main)
    error = np.max(np.abs(states[:, 0] - exact(t)))
    tol_errors.append(error)
    tol_residuals.append(abs(history[-1, 1]))
    print(f"{tol:.0e}       {abs(history[-1, 1]):.3e}      {error:.3e}      "
          f"{abs(s-s_exact):.3e}    {calls}")

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.loglog(tolerances, tol_errors, "o-", label="Максимальная ошибка угла")
ax.loglog(tolerances, tol_residuals, "s--", label="Конечная невязка |R_h|")
ax.invert_xaxis()
ax.set(xlabel="Допуск поиска корня (уменьшается вправо)", ylabel="Ошибка, рад",
       title="Уменьшение допуска при неизменном шаге h = 0.2")
ax.legend()
plt.show()

## Выводы

Точная начальная скорость равна $s_*\approx0.2199500341$.
На шаге $h=0.2$ оба метода дали $s_h\approx0.2199483608$.
Ньютон потребовал 4 вычисления невязки,
включая два вызова для производной, бисекция — 39.
Невязка линейна по $s$, поэтому Ньютон практически находит корень за одно обновление.
Такое преимущество связано с данной линейной задачей и не гарантировано в нелинейных задачах.

При уменьшении шага от 0.2 до 0.025 максимальная ошибка угла уменьшилась
с $3.379\cdot10^{-6}$ до $9.302\cdot10^{-10}$ рад.
Повторный подбор скорости на каждой сетке приближает её к точному значению.

Уменьшение допуска поиска корня снижает ошибку выполнения конечного условия
для численного интегратора, но не устраняет погрешность самого RK4.
При фиксированном шаге скорость подбирается к численному, а не точному решению задачи Коши.
Поэтому после достижения некоторого допуска ошибка траектории почти перестаёт уменьшаться.
Для дальнейшего повышения точности нужно уменьшить шаг интегрирования и заново выполнить стрельбу.